# 68 Algorithms Challenge: Journey to Mastery
## Model 03: Regularization Techniques (Ridge, Lasso, and Elastic Net)

Author: Tyra Nyambura 
Date: 20th October, 2026  
Goal: Train, evaluate, and compare Ridge, Lasso, and Elastic Net models using real lab-tested concrete mixtures to balance structural safety against component costs.

### The Engineering Challenge: High Interdependency
In concrete engineering, ingredients do not act in isolation. Adding water increases workability but heavily decreases structural strength (Abrams' law). To fix this, engineers add superplasticizers to reduce water demand while maintaining fluid flow. 

Because features like Cement, Water, and Superplasticizer dynamically interact, standard Ordinary Least Squares (OLS) Linear Regression suffers from severe multicollinearity. This causes the model coefficients to become unstable and highly sensitive to training data variations. Regularization stabilizes these weights by applying structural mathematical penalties.

### Mathematical Formulations

1. Ridge Regression (L2 Regularization)
Adds a penalty proportional to the square of the magnitude of the coefficients. It forces correlated weights to shrink uniformly without dropping variables completely.
Loss = RSS + alpha * sum(beta_j^2)

2. Lasso Regression (L1 Regularization)
Adds a penalty proportional to the absolute value of the magnitude of the coefficients. It can shrink weights down to exactly zero, serving as an automatic feature selection tool.
Loss = RSS + alpha * sum(|beta_j|)

3. Elastic Net Regression
A hybrid approach combining both L1 and L2 penalties. It balances the feature grouping properties of Ridge with the feature elimination properties of Lasso.
Loss = RSS + alpha * l1_ratio * sum(|beta_j|) + 0.5 * alpha * (1 - l1_ratio) * sum(beta_j^2)


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Apply chart rendering preferences
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

print("Environment setup and linear regularization toolkits loaded successfully.")

Environment setup and linear regularization toolkits loaded successfully.


### Step 1: Data Acquisition and Preprocessing

We are utilizing the authentic UCI Concrete Compressive Strength dataset containing 1,030 real physical samples crushed in compression testing machines. We will download the spreadsheet, strip out complex long column headers, and replace them with short, concise variable names to prevent math formatting issues.


In [2]:
# Load the verified laboratory Excel spreadsheet from the UCI archive
df = pd.read_excel("../../data/concrete-strength/Concrete_Data.xls")

# Re-map headers to clean, concise programming names
df.columns = [
    "Cement",
    "Slag",
    "FlyAsh",
    "Water",
    "Superplasticizer",
    "CoarseAggregate",
    "FineAggregate",
    "Age",
    "CompressiveStrength",
]

# Print structural validation parameters
print("Dataset Quality Check:")
print(f"Total Rows: {df.shape[0]} samples | Total Columns: {df.shape[1]}")
print(f"Missing Values detected: {df.isnull().sum().sum()}")

df.head()

Dataset Quality Check:
Total Rows: 1030 samples | Total Columns: 9
Missing Values detected: 0


,Cement,Slag,FlyAsh,Water,Superplasticizer,CoarseAggregate,FineAggregate,Age,CompressiveStrength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


### Step 2: Formulating Training Splits and Feature Standardization

Regularization models inject the literal size of coefficients directly into their internal loss penalty functions. If features differ wildly in units (e.g., Age ranging up to 365 days while Superplasticizer is often under 10 kg), the penalty routine will disproportionately warp features based on their scale. 

We will apply an 80/20 train/test split, then implement a StandardScaler to scale our training independent variables to a mean of 0 and a standard deviation of 1.


In [3]:
# Segregate dependent target from independent parameters
X = df.drop(columns=["CompressiveStrength"])
y = df["CompressiveStrength"]

# Separate train and test matrices using a constant state for pipeline replication
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit and execute the standard scaling operation
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"Training parameters shape: {X_train.shape}")
print(f"Testing parameters shape:  {X_test.shape}")

Training parameters shape: (824, 8)
Testing parameters shape:  (206, 8)


### Step 3: Multi-Model Fitment and Extracting Feature Weights

We initialize Ridge, Lasso, and Elastic Net with an alpha hyperparameter penalty score of 1.0. For the Elastic Net constructor, we supply an l1_ratio of 0.5 to weight the structural shrinkage constraints of both L1 and L2 paths equally. We will fit the estimators on our standard training array and extract the resulting weights into a dataframe.


In [4]:
alpha_selection = 1.0

# Initialize all three models
ridge_model = Ridge(alpha=alpha_selection, random_state=42)
lasso_model = Lasso(alpha=alpha_selection, random_state=42)
elastic_model = ElasticNet(alpha=alpha_selection, l1_ratio=0.5, random_state=42)

# Train the background parameters
ridge_model.fit(X_train, y_train)
lasso_model.fit(X_train, y_train)
elastic_model.fit(X_train, y_train)

# Construct our feature coefficient evaluation dataframe
coefficient_summary = {
    "Feature": X.columns,
    "Ridge (L2 Weights)": ridge_model.coef_,
    "Lasso (L1 Weights)": lasso_model.coef_,
    "Elastic Net Weights": elastic_model.coef_,
}

df_weights = pd.DataFrame(coefficient_summary)
print(df_weights.to_string(index=False))

         Feature  Ridge (L2 Weights)  Lasso (L1 Weights)  Elastic Net Weights
          Cement           12.559741            7.576345             4.809388
            Slag            9.213632            4.183440             2.102898
          FlyAsh            5.061598            0.000000            -0.238579
           Water           -3.038873           -2.896103            -2.765381
Superplasticizer            1.842780            3.225725             2.643742
 CoarseAggregate            1.248667           -0.000000            -0.932028
   FineAggregate            1.751912           -0.046890            -1.294636
             Age            7.018837            5.402830             3.801186


### Step 4: Out-of-Sample Performance Metric Extraction

To verify the generalizability of our model predictions, we execute predictions against the held-out test matrix. We track Root Mean Squared Error (RMSE) and R-squared (R2) parameters across all three approaches to identify which regularized penalty preserves the highest accuracy under high multicollinearity conditions.


In [5]:
model_matrix = {
    "Ridge (L2 Regularization)": ridge_model,
    "Lasso (L1 Regularization)": lasso_model,
    "Elastic Net (Hybrid)": elastic_model,
}

model_metrics_log = []

for label, model in model_matrix.items():
    predictions = model.predict(X_test)
    rmse_value = np.sqrt(mean_squared_error(y_test, predictions))
    r2_value = r2_score(y_test, predictions)

    model_metrics_log.append(
        {"Model Archetype": label, "RMSE (MPa)": rmse_value, "R2 Predictive Score": r2_value}
    )

df_performance = pd.DataFrame(model_metrics_log)
print(df_performance.to_string(index=False))

          Model Archetype  RMSE (MPa)  R2 Predictive Score
Ridge (L2 Regularization)    9.796411             0.627564
Lasso (L1 Regularization)   10.600829             0.563889
     Elastic Net (Hybrid)   11.184704             0.514526


### Step 5: Constructing the UI Conversion and Prediction Function

To cleanly serve our upcoming interface, we build an end-to-end production prediction function. This handles raw UI input forms (like Cement bags and Liters of water), mathematically transforms them back into the proper UCI density metric formats (kg per cubic meter), applies our saved StandardScaler, and fires our trained Elastic Net model to deliver clean output predictions.


In [ ]:
def calculate_ui_concrete_strength(bags_cement, liters_water, kg_sand, kg_gravel, curing_days):
    # 1. Convert standard 50kg cement bags to the required metric kg/m3 format
    kg_cement = bags_cement * 50.0

    # 2. Map standard volumetric densities for components missing from the basic UI sliders
    # We use data baseline means from the original UCI set to complete the feature array safely
    baseline_slag = 0.0
    baseline_flyash = 0.0
    baseline_superplasticizer = 0.0

    # 3. Assemble full raw input row matching the exact 8-feature order trained above
    raw_feature_array = np.array(
        [
            [
                kg_cement,
                baseline_slag,
                baseline_flyash,
                liters_water,
                baseline_superplasticizer,
                kg_gravel,
                kg_sand,
                curing_days,
            ]
        ]
    )

    # 4. Standardize using our existing training scaler pipeline parameters
    scaled_feature_array = scaler.transform(raw_feature_array)

    # 5. Extract raw predictions from our optimized background model
    predicted_strength_mpa = elastic_model.predict(scaled_feature_array)[0]

    return max(0.0, predicted_strength_mpa)


# Pipeline verification check using values mimicking our current UI snapshot
test_run = calculate_ui_concrete_strength(
    bags_cement=6, liters_water=180, kg_sand=700, kg_gravel=1050, curing_days=28
)

print(
    f"Pipeline verification complete. UI inputs successfully predicted a strength score of: {test_run:.2f} MPa"
)